In [1]:
%pip install opencv-python-headless

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os 
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

input_dir = r'drive-download-20240709T132713Z-001/Training'

imgDetails  = []

In [4]:
for className in os.listdir(input_dir):
    classPath = os.path.join(input_dir, className)

    for imageName in os.listdir(classPath):
        imagePath = os.path.join(classPath, imageName)

        image = cv2.imread(imagePath)

        if image is not None:
            height, width,channels = image.shape
            imageType = image.dtype
            resolution = (height, width)

            imgDetails.append({
                'imageName' : imageName,
                'class' : className,
                'height': height,
                'width' : width,
                'channels' : channels,
                'type' : imageType,
                'resolution' : resolution,
                'path' : imagePath
            })

imageDf = pd.DataFrame(imgDetails)

imageDf.to_csv('imgDetails.csv', index=False)

In [5]:
%pip install opencv-python numpy tqdm scikit-image

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install scikit-image


Note: you may need to restart the kernel to use updated packages.


In [7]:
import os 
import cv2
import numpy as np 
from tqdm import tqdm
from skimage import feature

In [8]:
def preImage(imagePath, outputPath, size=(224,224)):
    """
    Preprocess a single image by resizing, normalizing, and applying edge detection.

    Parameters:
    - image_path: str, path to the input image.
    - output_path: str, path to save the preprocessed image.
    - size: tuple, size to resize the image (default is (224, 224)).
    """


    img = cv2.imread(imagePath)
    if img is None:
        return
    
    imgResize = cv2.resize(img,size,interpolation=cv2.INTER_AREA)
    imgGray = cv2.cvtColor(imgResize, cv2.COLOR_BGR2GRAY)
    imgNormalized = imgGray/255.0   #normalizing to increase the contrast of the image 

    edge = feature.canny(imgNormalized, sigma=1)

    cv2.imwrite(outputPath, (edge * 255).astype(np.uint8))


In [9]:
def preData(input_dir, outputPath, size = (224,224)):
    """
    Preprocess all images in the dataset directory.

    Parameters:
    - input_dir: str, path to the input dataset directory.
    - output_dir: str, path to save the preprocessed dataset.
    - size: tuple, size to resize the images (default is (224, 224)).
    """

    if not os.path.exists(outputPath):
        os.makedirs(outputPath)

    classes = ['Empty', 'High','Low','Medium', 'Traffic Jam']

    for className in classes:
        inputClassDir = os.path.join(input_dir, className)
        outputClassDir = os.path.join(outputPath, className)

        if not os.path.exists(outputClassDir):
            os.makedirs(outputClassDir)

        if not os.path.exists(inputClassDir):
            print(f"Directory {inputClassDir} does not exitst skipping...")
            continue

        for imageName in tqdm(os.listdir(inputClassDir), desc=f'Preprocessing {className}'):
            inputImagePath = os.path.join(inputClassDir, imageName)
            outputImagePath = os.path.join(outputClassDir, imageName)
            preImage(inputImagePath, outputImagePath, size)


In [10]:
if __name__ =='__main__':
    input_dir = r'drive-download-20240709T132713Z-001/Training'
    outputPath = r'preprocessed data'
    preData(input_dir, outputPath)

Preprocessing Traffic Jam: 100%|██████████| 248/248 [00:02<00:00, 83.06it/s] 


In [11]:
%pip install tensorflow


Note: you may need to restart the kernel to use updated packages.


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt


In [14]:
basePath = 'preprocessed data'

classes = ['Empty', 'High', 'Low', 'Medium', 'Traffic Jam']

dataGen = ImageDataGenerator(rescale=1./255) # type: ignore

trainGen = dataGen.flow_from_directory(
    basePath,
    target_size=(224, 224),
    batch_size = 20,
    class_mode = 'categorical'
)



Found 3659 images belonging to 5 classes.
